# 🎬 AI Movie Translate & Dubbing Agent (v2.2) — Google Colab GPU Edition

Google Colab ပေါ်တွင် **NVIDIA T4 GPU (16GB VRAM)** ဖြင့် **Web UI Dashboard** ကို 1-Click အလွယ်တကူ ဖွင့်လှစ်အသုံးပြုနိုင်သော စနစ် ဖြစ်ပါသည်။

### 🌟 v2.2 စနစ်သစ် အဓိက လုပ်ဆောင်ချက်များ:
* 🍪 **Multi-Platform Auto Downloader:** YouTube (Anti-Bot Bypass), DramaBox (`dramaboxdb.com`), ReelShort (`reelshort.com`) link များ တိုက်ရိုက်ဒေါင်းလုဒ်ဆွဲနိုင်ခြင်း။
* 🛑 **1-Click Force Stop Pipeline:** ဗီဒီယိုထုတ်လုပ်နေစဉ် မည်သည့်အချိန်မဆို ချက်ချင်း ရပ်တန့်နိုင်ခြင်း (Instant VRAM & CPU release)။
* 📝 **Subtitle Mode Switch:** Hardsub မြန်မာစာတန်းထိုး တိုက်ရိုက်ထည့်ခြင်း (သို့မဟုတ်) စာတန်းထိုး မပါဘဲ Voiceover Only ထုတ်ယူခြင်း (Standalone `.srt` & `.ass` ပါဝင်)။
* ⚙️ **Resolution Quality Presets:** 1080p Full HD (Crisp / အကြည်လင်ဆုံး) နှင့် 720p HD (Faster Render / အမြန်ဆုံး)။
* 📱 **Facebook Reels & TikTok (9:16 Canvas Exporter):** 1080x1920 Full HD Canvas နှင့် Silky Blurred Background ကို ၄၅ စက္ကန့်အတွင်း အမြန်ဆုံး ထုတ်ပေးခြင်း။
* 👫 **AI Multi-Voice & Smart Script:** Thiha (ကျား) နှင့် Nilar (မ) အသံခွဲဝေမှု + စကားမပြောသော အသံတိတ်ခန်း (>18s) များတွင် ဇာတ်လမ်းဆက်ပြောပေးခြင်း (Action Narration Bridge)။
* ☁️ **Permanent Google Drive Sync:** Google Drive ချိတ်ဆက်ထားပါက Database, Cookies, Config နှင့် ဗီဒီယိုများ အလိုအလျောက် ထာဝရ သိမ်းဆည်းပေးခြင်း။

---

In [ ]:
# @title 🚀 1. Launch Web UI Dashboard (1-Click & Permanent Google Drive Sync)
# @markdown အောက်ပါ Checkbox ကို အမှန်ခြစ်ထားပါက ထွက်ရှိလာသော ဗီဒီယိုများ၊ Database နှင့် Cookies များကို မိမိ Google Drive ထဲသို့ အလိုအလျောက် ထာဝရ သိမ်းဆည်းပေးပါမည်:
mount_google_drive = True #@param {type:"boolean"}

import os, sys, subprocess, time, re, shutil, socket, torch
from IPython.display import display, HTML, Javascript

# 0. GPU Hardware Verification (CUDA & VRAM Check)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'🚀 [GPU Active] {gpu_name} ({vram_gb:.1f} GB VRAM) — Hardware Acceleration 100% Ready!')
else:
    print('⚠️ [CPU Notice] No GPU detected! Go to: Runtime > Change runtime type > T4 GPU for 10x faster rendering.')

# 1. Background Keep-Alive Heartbeat (Colab လိုင်းမပြတ်စေရန် ကာကွယ်ခြင်း)
try:
    display(Javascript('''
    setInterval(function(){
        try {
            var btn = document.querySelector("colab-connect-button");
            if (btn && btn.shadowRoot) {
                var connectBtn = btn.shadowRoot.querySelector("#connect");
                if (connectBtn) connectBtn.click();
            }
        } catch(e){}
    }, 60000);
    '''))
    print("⚡ Colab Auto-KeepAlive Active (Prevents idle disconnect)")
except Exception:
    pass

# 2. Optional: Google Drive Permanent Storage Mount
drive_outputs_dir = "/content/drive/MyDrive/MovieRecapOutputs"
if mount_google_drive:
    try:
        from google.colab import drive
        print("[*] Mounting Google Drive for permanent output & database storage...")
        drive.mount('/content/drive', force_remount=False)
        os.makedirs(drive_outputs_dir, exist_ok=True)
        print(f"[OK] Google Drive Connected! Saved to: {drive_outputs_dir}")
    except Exception as e:
        print(f"[WARN] Google Drive mount skipped or failed ({e}). Using local VM storage.")

project_dir = "/content/ai-translate-agent"

# 3. Setup Repository (အသစ်ဆုံး Version သို့ အလိုအလျောက် Update လုပ်ခြင်း)
if not os.path.exists(project_dir):
    print("[*] 1/4 Cloning repository...")
    !git clone https://github.com/paipai1999/ai-translate-agent.git {project_dir}
else:
    print("[*] 1/4 Updating repository to latest commit...")
    !cd {project_dir} && git fetch origin main && git reset --hard origin/main

os.chdir(project_dir)
%cd /content/ai-translate-agent

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

# Create config.json from config.example.json if missing
cfg_path = os.path.join(project_dir, "config.json")
cfg_example = os.path.join(project_dir, "config.example.json")
if not os.path.exists(cfg_path) and os.path.exists(cfg_example):
    shutil.copy(cfg_example, cfg_path)

# Restore permanent config.json, database.db & cookies.txt from Google Drive if available
if mount_google_drive and os.path.exists(drive_outputs_dir):
    drive_cfg = os.path.join(drive_outputs_dir, "config.json")
    if os.path.exists(drive_cfg):
        shutil.copy2(drive_cfg, cfg_path)
        print("🔑 [Drive Sync] config.json (API keys & settings) restored from Google Drive!")
    for db_name in ["movie_metadata.db", "database.db"]:
        d_db = os.path.join(drive_outputs_dir, db_name)
        if os.path.exists(d_db):
            shutil.copy2(d_db, os.path.join(project_dir, db_name))
            os.makedirs(os.path.join(project_dir, "outputs"), exist_ok=True)
            shutil.copy2(d_db, os.path.join(project_dir, "outputs", db_name))
            print(f"🗄️ [Drive Sync] {db_name} restored from Google Drive!")
    drive_cookies = os.path.join(drive_outputs_dir, "cookies.txt")
    if os.path.exists(drive_cookies):
        shutil.copy2(drive_cookies, "cookies.txt")
        shutil.copy2(drive_cookies, os.path.join("assets", "cookies.txt"))
        print("🍪 [Drive Sync] cookies.txt restored from Google Drive!")

os.makedirs("temp", exist_ok=True)
os.makedirs("movies", exist_ok=True)

# Safe Symlink outputs to Google Drive (Zero Symlink Conflicts)
if mount_google_drive and os.path.exists(drive_outputs_dir):
    if os.path.islink("outputs"):
        try: os.unlink("outputs")
        except Exception: pass
    elif os.path.isdir("outputs"):
        for item in os.listdir("outputs"):
            src = os.path.join("outputs", item)
            dst = os.path.join(drive_outputs_dir, item)
            if not os.path.exists(dst):
                if os.path.isdir(src): shutil.copytree(src, dst)
                else: shutil.copy2(src, dst)
        shutil.rmtree("outputs", ignore_errors=True)
    if not os.path.exists("outputs"):
        os.symlink(drive_outputs_dir, "outputs")
    print("📁 [Drive Sync Active] outputs/ is permanently linked to Google Drive!")
else:
    os.makedirs("outputs", exist_ok=True)

# 4. Install System Dependencies & Myanmar Padauk Fonts
print("[*] 2/4 Checking & Installing system dependencies...")
if not os.path.exists("/usr/share/fonts/truetype/padauk/Padauk.ttf"):
    !apt-get update -qq && apt-get install -y -qq ffmpeg fonts-sil-padauk fonts-noto-cjk fonts-noto-core > /dev/null 2>&1
    !fc-cache -f > /dev/null 2>&1

# Install GPU-accelerated NVENC FFmpeg if running on CUDA
if torch.cuda.is_available() and not os.path.exists("/usr/local/bin/ffmpeg"):
    try:
        chk = subprocess.run(["ffmpeg", "-encoders"], capture_output=True, text=True)
        if "h264_nvenc" not in chk.stdout:
            print("[*] Installing NVIDIA NVENC GPU-accelerated FFmpeg...")
            !wget -q -nc https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz -O /tmp/ffmpeg.tar.xz && tar -xf /tmp/ffmpeg.tar.xz -C /tmp && cp /tmp/ffmpeg-*/bin/ffmpeg /usr/local/bin/ffmpeg && cp /tmp/ffmpeg-*/bin/ffprobe /usr/local/bin/ffprobe && rm -rf /tmp/ffmpeg*
            print("🚀 [OK] NVIDIA NVENC FFmpeg ready!")
    except Exception as e:
        print(f"[WARN] NVENC FFmpeg check notice: {e}")

# Ensure yt-dlp and requirements are installed
!pip install -q -U yt-dlp
!pip install -q -r requirements.txt

# Install Cloudflared Tunnel
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Stop any existing server processes
!pkill -f "web_ui.py" || true
!pkill -f "cloudflared" || true

# 5. Start Web UI Server
print("[*] 3/4 Starting Web UI Server...")
server_proc = subprocess.Popen(
    [sys.executable, "web_ui.py", "--host", "0.0.0.0", "--port", "5000"],
    cwd=project_dir,
    stdout=open("/content/web_ui.log", "w"),
    stderr=subprocess.STDOUT
)

# Wait until Web UI port 5000 is actively accepting connections
server_ready = False
for _ in range(35):
    if server_proc.poll() is not None:
        break
    try:
        with socket.create_connection(("127.0.0.1", 5000), timeout=1):
            server_ready = True
            break
    except OSError:
        time.sleep(1)

if not server_ready:
    print("❌ Web UI failed to start! Crash log details:")
    if os.path.exists("/content/web_ui.log"):
        with open("/content/web_ui.log", "r") as f:
            print(f.read())
else:
    # Get Google Colab Native Proxy Port
    colab_native_url = None
    try:
        from google.colab.output import eval_js
        colab_native_url = eval_js("google.colab.kernel.proxyPort(5000)")
    except Exception:
        pass

    print("[*] 4/4 Connecting Cloudflare Secure Tunnel...")
    tunnel_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    tunnel_url = None
    start_t = time.time()
    while time.time() - start_t < 40:
        line = tunnel_proc.stdout.readline()
        if not line and tunnel_proc.poll() is not None:
            break
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

    # Allow 3 seconds for Cloudflare Global DNS to propagate
    time.sleep(3)

    primary_url = colab_native_url or tunnel_url
    
    native_btn_html = f'''
    <a href="{colab_native_url}" target="_blank" style="background: linear-gradient(135deg, #1f6feb, #238636); color: #ffffff; font-weight: bold; font-size: 18px; padding: 14px 30px; border-radius: 10px; text-decoration: none; display: inline-block; box-shadow: 0 4px 16px rgba(31, 111, 235, 0.4); margin: 6px;">
        ⚡ Open via Google Colab Direct ↗️
    </a>
    ''' if colab_native_url else ''

    tunnel_btn_html = f'''
    <a href="{tunnel_url}" target="_blank" style="background: linear-gradient(135deg, #f0883e, #da3633); color: #ffffff; font-weight: bold; font-size: 18px; padding: 14px 30px; border-radius: 10px; text-decoration: none; display: inline-block; box-shadow: 0 4px 16px rgba(240, 136, 62, 0.4); margin: 6px;">
        ☁️ Open via Cloudflare Tunnel ↗️
    </a>
    ''' if tunnel_url else ''

    if primary_url:
        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #0d1117, #161b22); border: 2px solid #58a6ff; border-radius: 14px; padding: 26px; text-align: center; margin: 20px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow: 0 8px 24px rgba(0,0,0,0.5);">
            <div style="font-size: 38px; margin-bottom: 6px;">🎬</div>
            <h2 style="color: #58a6ff; margin: 0 0 10px 0; font-size: 22px;">Web UI Dashboard အဆင်သင့် ဖြစ်ပါပြီ!</h2>
            <p style="color: #c9d1d9; font-size: 14px; margin: 0 0 18px 0;">အောက်ပါ ခလုတ်များအနက် အဆင်ပြေရာတစ်ခုကို နှိပ်၍ Web UI ကို ဖွင့်ပါ 👇</p>
            <div style="display: flex; justify-content: center; flex-wrap: wrap; gap: 10px;">
                {native_btn_html}
                {tunnel_btn_html}
            </div>
            <div style="margin-top: 16px; font-size: 13px; color: #8b949e;">
                💡 <i>Cloudflare တွင် 'DNS_PROBE_FINISHED_NXDOMAIN' ပေါ်ပါက ၅ စက္ကန့် စောင့်ပြီး <b>Reload</b> နှိပ်ပါ သို့မဟုတ် <b>Google Colab Direct</b> ခလုတ်ကို အသုံးပြုပါ။</i>
            </div>
        </div>
        """))
        print(f"\n👉 Primary Direct URL: {primary_url}")
        if tunnel_url: print(f"👉 Cloudflare URL: {tunnel_url}")


In [ ]:
# @title 💻 2. Command Line (CLI) Direct Run (Alternative)
# @markdown Web UI မသုံးဘဲ Notebook ထဲမှ တိုက်ရိုက် Recap ဗီဒီယို ထုတ်ယူလိုပါက အောက်ပါအတိုင်း ဖြည့်သွင်း၍ Run နိုင်ပါသည်:
video_input = "https://youtu.be/KmYSM5knNV8" #@param {type:"string"}
thumbnail_title = "" #@param {type:"string"}
subtitle_mode = "burn" #@param ["burn", "none"]
resolution = "1080p" #@param ["1080p", "720p"]

cmd = f"python main.py --input '{video_input}' --sub-mode '{subtitle_mode}' --resolution '{resolution}'"
if thumbnail_title.strip():
    cmd += f" --thumb-title '{thumbnail_title.strip()}'"

!{cmd}


In [ ]:
# @title 📁 3. View & Download Generated Outputs
import glob
from IPython.display import FileLink

output_files = glob.glob("outputs/**/*.mp4", recursive=True) + glob.glob("/content/drive/MyDrive/MovieRecapOutputs/**/*.mp4", recursive=True)
output_files = sorted(list(set(output_files)))

if not output_files:
    print("No output videos found yet. Run the Web UI or CLI above to generate recaps!")
else:
    print(f"Found {len(output_files)} generated video(s):")
    for f in output_files:
        size_mb = os.path.getsize(f) / (1024*1024) if os.path.exists(f) else 0
        print(f"🎬 {f} ({size_mb:.1f} MB)")
